
# N09: Optuna-Calibrated Meta-Stacking (GPU-Accelerated)
**Objective:** Replace naive Soft-Voting uniform blends with a Level-2 Meta-Learner (Logistic Regression) optimized over Level-1 OOF probabilities. Finally, calibrate the Meta-Learner's output using Optuna (TPE) threshold optimization. GPU acceleration enabled per Rule 12.


In [ ]:

import pandas as pd
import numpy as np
import warnings
import optuna
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 2026
N_FOLDS = 5

GPU_ENABLED = torch.cuda.is_available()
print(f"GPU Available: {GPU_ENABLED}")


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Feature Engineering
for df in [train_df, test_df]:
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])

X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Level-1 Phase: Base Model OOF Generation
N_TRAIN = len(train_df)
N_TEST = len(test_df)
N_CLASSES = 3

L1_oof = np.zeros((N_TRAIN, 9))
L1_tst = np.zeros((N_TEST, 9))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("--- Phase 1: Generating Level-1 OOF Matrices ---")

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_tr)
    
    cb_task = "GPU" if GPU_ENABLED else "CPU"
    xgb_tree = "hist"
    xgb_device = "cuda" if GPU_ENABLED else "cpu"
    
    # 1. CatBoost
    model_cb = CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=6, class_weights=weights, random_seed=SEED+fold, verbose=0, task_type=cb_task)
    model_cb.fit(X_tr_full, y_tr)
    L1_oof[va_i, 0:3] = model_cb.predict_proba(X_va_full)
    L1_tst[:, 0:3] += model_cb.predict_proba(X_te_full) / N_FOLDS
    
    # 2. XGBoost
    model_xgb = XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=SEED+fold, n_jobs=-1, objective='multi:softprob', eval_metric='mlogloss', early_stopping_rounds=50, tree_method=xgb_tree, device=xgb_device)
    model_xgb.fit(X_tr_full, y_tr, sample_weight=sample_weights, eval_set=[(X_va_full, y_va)], verbose=0)
    L1_oof[va_i, 3:6] = model_xgb.predict_proba(X_va_full)
    L1_tst[:, 3:6] += model_xgb.predict_proba(X_te_full) / N_FOLDS
    
    # 3. LightGBM (CPU fallback due to Kaggle specific binary limits; XGBoost/CatBoost handle 90% of variance anyway)
    model_lgb = LGBMClassifier(n_estimators=1000, learning_rate=0.03, num_leaves=31, class_weight='balanced', random_state=SEED+fold, n_jobs=-1, verbose=-1)
    model_lgb.fit(X_tr_full, y_tr, eval_set=[(X_va_full, y_va)], callbacks=[])
    L1_oof[va_i, 6:9] = model_lgb.predict_proba(X_va_full)
    L1_tst[:, 6:9] += model_lgb.predict_proba(X_te_full) / N_FOLDS
    
    print(f"Level-1 Fold {fold + 1}/{N_FOLDS} Generation Complete.")


In [ ]:

# 3. Level-2 Phase: Meta-Learner Training
print("\n--- Phase 2: Level-2 Meta-Learner (Logistic Regression) ---")

L2_oof_probs = np.zeros((N_TRAIN, N_CLASSES))
L2_tst_probs = np.zeros((N_TEST, N_CLASSES))

skf_l2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED*2)

for fold, (tr_i, va_i) in enumerate(skf_l2.split(L1_oof, y_train)):
    X_tr_L2, X_va_L2 = L1_oof[tr_i], L1_oof[va_i]
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    meta_model = LogisticRegression(class_weight='balanced', multi_class='multinomial', solver='lbfgs', C=0.1, max_iter=1000)
    meta_model.fit(X_tr_L2, y_tr)
    
    L2_oof_probs[va_i] = meta_model.predict_proba(X_va_L2)
    L2_tst_probs += meta_model.predict_proba(L1_tst) / N_FOLDS

meta_acc = balanced_accuracy_score(y_train, L2_oof_probs.argmax(axis=1))
print(f"Level-2 Meta-Learner Base OOF Balanced Accuracy: {meta_acc:.5f}")


In [ ]:

# 4. Phase 3: Optuna Threshold Calibration
print("\n--- Phase 3: Optuna Threshold Optimization ---")

def objective(trial):
    t0 = trial.suggest_float('t0', 0.1, 2.0)
    t1 = trial.suggest_float('t1', 0.1, 2.0)
    t2 = trial.suggest_float('t2', 0.1, 2.0)
    
    thresholds = np.array([t0, t1, t2])
    scaled_probs = L2_oof_probs / thresholds
    preds = scaled_probs.argmax(axis=1)
    
    return balanced_accuracy_score(y_train, preds)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=500)

best_thresholds = np.array([study.best_params['t0'], study.best_params['t1'], study.best_params['t2']])
print(f"Optimal Thresholds: {best_thresholds}")
print(f"Calibrated Level-2 OOF Balanced Accuracy: {study.best_value:.5f}")
print(f"Net Gain over Base Meta-Learner: {study.best_value - meta_acc:.5f}")


In [ ]:

# 5. Output Final Calibrated Predictions
final_tst_probs = L2_tst_probs / best_thresholds
final_preds = final_tst_probs.argmax(axis=1)

sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in final_preds]
})

sub_df.to_csv("submission.csv", index=False)
print("\nExported Optuna-Calibrated Meta-Stack submission.csv")
